# Medallion Architecture in Databricks with TPC-H Samples

This notebook implements the Medallion Architecture using the built-in samples.tpch dataset. It creates Bronze, Silver, and Gold layers in your Databricks workspace.

## Step 0: Create Catalog (for Unity Catalog-enabled workspaces)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS main;

## Step 1: Set Up Schemas

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS main.bronze;
CREATE SCHEMA IF NOT EXISTS main.silver;
CREATE SCHEMA IF NOT EXISTS main.gold;

## Step 2: Bronze Layer - Ingest Raw Data

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Orders table
orders_df = spark.table("samples.tpch.orders")
bronze_orders = orders_df.withColumn("ingest_ts", current_timestamp()).withColumn("source", lit("tpch"))
bronze_orders.write.format("delta").mode("overwrite").saveAsTable("main.bronze.orders")

# Lineitem table
lineitem_df = spark.table("samples.tpch.lineitem")
bronze_lineitem = lineitem_df.withColumn("ingest_ts", current_timestamp()).withColumn("source", lit("tpch"))
bronze_lineitem.write.format("delta").mode("overwrite").saveAsTable("main.bronze.lineitem")

# Customer table
customer_df = spark.table("samples.tpch.customer")
bronze_customer = customer_df.withColumn("ingest_ts", current_timestamp()).withColumn("source", lit("tpch"))
bronze_customer.write.format("delta").mode("overwrite").saveAsTable("main.bronze.customer")

# Nation table
nation_df = spark.table("samples.tpch.nation")
bronze_nation = nation_df.withColumn("ingest_ts", current_timestamp()).withColumn("source", lit("tpch"))
bronze_nation.write.format("delta").mode("overwrite").saveAsTable("main.bronze.nation")

## Step 3: Silver Layer - Clean and Standardize Data

In [0]:
from pyspark.sql.functions import col, to_date

# Orders table
bronze_orders = spark.table("main.bronze.orders")
silver_orders = bronze_orders \
    .withColumnRenamed("o_orderkey", "order_key") \
    .withColumnRenamed("o_custkey", "customer_key") \
    .withColumnRenamed("o_orderstatus", "order_status") \
    .withColumnRenamed("o_totalprice", "total_price") \
    .withColumn("order_date", to_date(col("o_orderdate"), "yyyy-MM-dd")) \
    .drop("o_orderdate") \
    .dropDuplicates(["order_key"])
silver_orders.write.format("delta").mode("overwrite").saveAsTable("main.silver.orders")

# Lineitem table
bronze_lineitem = spark.table("main.bronze.lineitem")
silver_lineitem = bronze_lineitem \
    .withColumnRenamed("l_orderkey", "order_key") \
    .withColumnRenamed("l_extendedprice", "extended_price") \
    .withColumn("extended_price", col("extended_price").cast("decimal(18,2)")) \
    .dropDuplicates(["order_key", "l_linenumber"])
silver_lineitem.write.format("delta").mode("overwrite").saveAsTable("main.silver.lineitem")

## Step 4: Gold Layer - Create Business-Ready Aggregates

In [0]:
from pyspark.sql.functions import sum

# Revenue by nation
silver_orders = spark.table("main.silver.orders")
silver_lineitem = spark.table("main.silver.lineitem")
bronze_customer = spark.table("main.bronze.customer")  # Using bronze for simplicity
bronze_nation = spark.table("main.bronze.nation")

gold_revenue = silver_lineitem.alias("l") \
    .join(silver_orders.alias("o"), col("l.order_key") == col("o.order_key")) \
    .join(bronze_customer.alias("c"), col("o.customer_key") == col("c.c_custkey")) \
    .join(bronze_nation.alias("n"), col("c.c_nationkey") == col("n.n_nationkey")) \
    .groupBy("n.n_name") \
    .agg(sum(col("l.extended_price") * (1 - col("l.l_discount"))).alias("total_revenue")) \
    .orderBy(col("total_revenue").desc())
gold_revenue.write.format("delta").mode("overwrite").saveAsTable("main.gold.revenue_by_nation")

# Display sample
display(spark.table("main.gold.revenue_by_nation").limit(5))

n_name,total_revenue
IRAQ,44199396357.7281
INDONESIA,44156732316.9634
GERMANY,43966473754.5423
FRANCE,43954896466.8247
IRAN,43951480204.7432


## Verification Queries

In [0]:
%sql
SELECT * FROM main.bronze.orders LIMIT 5;

o_orderkey,o_custkey,o_orderstatus,o_totalprice,o_orderdate,o_orderpriority,o_clerk,o_shippriority,o_comment,ingest_ts,source
5611649,687736,O,51905.72,1996-06-04,5-LOW,Clerk#000002954,0,"ls! bold, regular accounts",2025-10-26T13:34:29.931Z,tpch
5611650,513292,O,62845.59,1997-02-28,5-LOW,Clerk#000002092,0,"to boost ironically regular deposits. unusual, ironic exc",2025-10-26T13:34:29.931Z,tpch
5611651,395308,F,226256.25,1992-03-05,1-URGENT,Clerk#000004123,0,its. regular foxes boost speci,2025-10-26T13:34:29.931Z,tpch
5611652,423847,O,141103.54,1995-08-07,3-MEDIUM,Clerk#000003975,0,onic accounts. furiously permanent foxes slee,2025-10-26T13:34:29.931Z,tpch
5611653,90844,O,157430.11,1996-08-14,5-LOW,Clerk#000004721,0,the carefully even th,2025-10-26T13:34:29.931Z,tpch


In [0]:
%sql
SELECT * FROM main.silver.orders LIMIT 5;

order_key,customer_key,order_status,total_price,o_orderpriority,o_clerk,o_shippriority,o_comment,ingest_ts,source,order_date
13723365,224620,O,147726.24,5-LOW,Clerk#000000132,0,blithely. furiously final sheaves ha,2025-10-26T13:34:29.931Z,tpch,1996-01-13
13741954,271702,O,21226.43,4-NOT SPECIFIED,Clerk#000003188,0,"ependencies sleep carefully regular foxes. regular, fi",2025-10-26T13:34:29.931Z,tpch,1998-04-16
13743302,232108,O,278875.33,1-URGENT,Clerk#000000639,0,dolites alongside of the express ins,2025-10-26T13:34:29.931Z,tpch,1996-09-20
13744258,270794,F,190350.13,4-NOT SPECIFIED,Clerk#000001680,0,mong the furiously ev,2025-10-26T13:34:29.931Z,tpch,1993-07-27
13747588,207142,F,129293.64,3-MEDIUM,Clerk#000000271,0,eodolites. furiously final asymptotes sleep beyond the carefu,2025-10-26T13:34:29.931Z,tpch,1993-05-06


In [0]:
%sql
SELECT * FROM main.gold.revenue_by_nation LIMIT 5;

n_name,total_revenue
IRAQ,44199396357.7281
INDONESIA,44156732316.9634
GERMANY,43966473754.5423
FRANCE,43954896466.8247
IRAN,43951480204.7432
